# Pipeline: compartment calling

Part of the **[Fig. 4 chapter](fig4.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


## 📥 Required input files

This notebook reads the following files (paths resolve from `ENTEX_ROOT`/`REF_ROOT`; the setup cell sets them). See the chapter's `inputs.md` for RAW-vs-derived tags.

- `f'{outdir}merged_comp.hdf'`  ·  _other_
- `f'{REF_ROOT}/hg38/hg38.100kbin.CpG.txt'`  ·  _reference_
- `f'{indir_raw}merged.mcool::resolutions/100000'`  ·  _other_
- `f'{indir_impute}merged.cool'`  ·  _contacts (cool)_
- `f'{ENTEX_ROOT}/L1color.tsv'`  ·  _metadata: color_
- `f'{indir}{ct}/{ct}.Q.cool'`  ·  _contacts (cool)_
- `f'{indir}{ct}/{ct}/{ct}.raw.mcool::resolutions/100000'`  ·  _other_


In [1]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import os
import time
import cooler
from glob import glob
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
from concurrent.futures import ProcessPoolExecutor, as_completed

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [3]:
indir_raw = f'{ENTEX_ROOT}/merged_cool_raw/'
indir_impute = f'{ENTEX_ROOT}/merged_cool_impute/100K/'
outdir = f'{ENTEX_ROOT}/analysis/compartment/'


In [4]:
# indir = indir_impute
indir = indir_raw
coollist = glob(f'{indir}L2any/c*-c*.cool')
print(len(coollist))


In [5]:
cool_df = pd.DataFrame(coollist, columns=['cool_path'])
cool_df['L2any'] = [xx.split('/')[-1].split('.')[0] for xx in cool_df['cool_path'].values]
cool_df['L1'] = cool_df['L2any'].str.split('-').str[0]
cool_df.loc[cool_df['L1']=='c7', 'L1'] = 'c35'
cool_df.loc[(cool_df['L1']=='c35') & cool_df['L2any'].str.split('-').str[1].isin(['c0','c10','c11']), 'L1'] = 'c36'


In [6]:
def merge_impute_cool(group, cool_path):
    # os.makedirs(f'{indir}L1/', exist_ok=True)
    cool_list_path = f'{indir}L1/coollist_{group}.txt'
    cool_path.to_csv(cool_list_path, header=False, index=False)
    cmd = f'hicluster merge-cool --input_cool_tsv_file {cool_list_path} --output_cool {indir}L1/{group}.cool'
    os.system(cmd)
    return f'{indir}L1/{group}.cool'

def merge_raw_cool(group, cool_path):
    # os.makedirs(f'{outdir}chunk{group}/', exist_ok=True)
    cmd = f'cooler merge {indir}L1/{group}.cool ' + ' '.join(list(cool_path))
    os.system(cmd)
    return f'{indir}L1/{group}.cool'


In [7]:
from concurrent.futures import ProcessPoolExecutor, as_completed

cpu = 30
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for group,df in cool_df.loc[cool_df['L1'].isin(['c35','c36'])].groupby('L1'):
        future = executor.submit(
            merge_raw_cool,
            group=group,
            cool_path=df['cool_path'],
        )
        futures[future] = group

    result = []
    for future in as_completed(futures):
        group = futures[future]
        result.append(future.result())
        print(f'{group} finished')
        

In [8]:
## raw
# cmd = f'cooler merge {outdir}merged.raw.cool ' + ' '.join(list(result))
# os.system(cmd)
# cmd = f'cooler zoomify -r 5000N -n 32 {outdir}merged.raw.cool'
# os.system(cmd)


In [9]:
## impute
cool_list_path = f'{indir}coollist.txt'
np.savetxt(cool_list_path, result, fmt='%s', delimiter='\n')
cmd = f'hicluster merge-cool --input_cool_tsv_file {cool_list_path} --output_cool {indir}merged.cool'
os.system(cmd)


In [10]:
res = 100000
chrom_size_path = '/large_experiments/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [11]:
if os.path.isfile(f'{outdir}merged_comp.hdf'):
    bin_df = pd.read_hdf(f'{outdir}merged_comp.hdf', key='data')
else:
    bin_df = cooler.util.binnify(chrom_sizes, res)
    bin_df['chrom'] = bin_df['chrom'].astype(str)

bin_df.shape

In [12]:
cpg = pd.read_csv(f'{REF_ROOT}/hg38/hg38.100kbin.CpG.txt', header=0, index_col=3, sep='\t')
cpg['CpG_density'] = cpg['14_user_patt_count'] / (cpg['13_seq_len'] - cpg['11_num_N'])


## Fit PC model with merged raw

In [14]:
Qall = []
binall = []
mode = 'impute'
if mode=='raw':
    cool = cooler.Cooler(f'{indir_raw}merged.mcool::resolutions/100000')
elif mode=='impute':
    cool = cooler.Cooler(f'{indir_impute}merged.cool')
fig, axes = plt.subplots(5, 5, figsize=(15,15))
for i,c in enumerate(chrom_sizes.index):
    Q = cool.matrix(balance=False, sparse=True).fetch(c).toarray()
    Q = Q - np.diag(np.diag(Q))
    rowsum = Q.sum(axis=0)
    thres = [np.percentile(rowsum[rowsum>0], 50), np.percentile(rowsum[rowsum>0], 99)]
    thres.append(thres[0]*2-thres[1])
    ax = axes.flatten()[i]
    sns.histplot(rowsum, bins=100, ax=ax)
    for t in thres:
        ax.plot([t, t], [0, ax.get_ylim()[1]*0.5], c='r')
    ax.set_title(c)
    binfilter = (rowsum>thres[-1])
    binall.append(binfilter)
    Q = Q[binfilter][:, binfilter]
    Qall.append(Q)
    print(c)

for i in range(chrom_sizes.shape[0], axes.flatten().shape[0]):
    axes.flatten()[i].axis('off')
    

In [15]:
# coollist = pd.read_csv(f'{ENTEX_ROOT}/clustering/merged/coollist_group.csv', header=None, index_col=0)
# coollist.columns = ['coolpath', 'group']
# coollist['L2any'] = ['-'.join(xx.split('-')[:2]) for xx in coollist['group'].values]
# coollist['L1'] = coollist['L2any'].str.split('-').str[0]

In [16]:
# for xx,df in coollist.groupby('L2any'):
#     cool = cooler.Cooler(f'{indir}L2any/{xx}.cool')
#     if df.shape[0]!=cool.info['group_n_cells']:
#         print(xx, df.shape[0])


In [17]:
# tot = 0
# for xx,df in coollist.groupby('L1'):
#     cool = cooler.Cooler(f'{indir}L1/{xx}.cool')
#     tot += cool.info['group_n_cells']
#     if df.shape[0]!=cool.info['group_n_cells']:
#         print(xx, df.shape[0], cool.info['group_n_cells'])

# print(tot)

In [18]:
cool.info

In [19]:
fig, axes = plt.subplots(5, 5, figsize=(15,15))
for i,c in enumerate(chrom_sizes.index):
    ax = axes.flatten()[i]
    n_bins = int(chrom_sizes[c] // 100000) + 1
    tmp = np.zeros((n_bins, n_bins))
    tmp[np.ix_(binall[i], binall[i])] = Qall[i]
    ax.imshow(tmp, cmap='bwr', vmin=-np.percentile(Qall[i], 95), vmax=np.percentile(Qall[i], 95))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(c, fontsize=15)

for i in range(chrom_sizes.shape[0], axes.flatten().shape[0]):
    axes.flatten()[i].axis('off')
    
plt.tight_layout()
# plt.savefig(f'{indir}/plot/celltype_A_decay.pdf', transparent=True)


In [20]:
# comp = pd.read_hdf(f'{indir}compartment_majortype/comp_raw_merged.hdf', key='data')

Call = []
pcall = []
modelall = []
for k,chrom in enumerate(chrom_sizes.index):
    Q = Qall[k].copy()
    decay = np.array([np.mean(np.diag(Q, i)) for i in range(Q.shape[0])])
    E = np.zeros(Q.shape)
    row, col = np.diag_indices(E.shape[0])
    E[row, col] = 1
    for i in range(1, E.shape[0]):
        E[row[:-i], col[i:]] = (Q[row[:-i], col[i:]] + 1e-5) / (decay[i] + 1e-5)
    E = E + E.T
    C = np.corrcoef(np.log2(E + 0.001))
    vmin, vmax = np.percentile(C, 5), np.percentile(C, 95)
    C[C<vmin] = vmin
    C[C>vmax] = vmax
    Call.append(C)
    binfilter = binall[k]
#     tmp = comp.loc[ct, (comp.columns.str.split('_').str[0]==chrom)]
#     tmp.index = [int(xx.split('_')[1]) for xx in tmp.index]
#     pcall.append(tmp[np.where(binfilter)[0]].values)
    pca = PCA(n_components=2, svd_solver='arpack')
    pc = pca.fit_transform(C)
    cpgtmp = cpg.loc[cpg['#1_usercol']==chrom, 'CpG_density'].values[binfilter]
    # i = 0
    # if np.abs(pearsonr(cpgtmp, pc[:,0])[0])>np.abs(pearsonr(cpgtmp, pc[:,1])[0]):
    #     i = 0
    # else:
    #     i = 1
    r = []
    for i in range(2):
        labels, groups = pd.qcut(pc[:,i], 50, labels=False, retbins=True)
        sad = np.array([[E[np.ix_(labels==i, labels==j)].sum() for i in range(50)] for j in range(50)])
        count = np.array([[(labels==i).sum()*(labels==j).sum() for i in range(50)] for j in range(50)])
        sad = sad / count
        r.append((sad[:10, :10].sum() + sad[-10:, -10:].sum()) / (sad[:10, -10:].sum() + sad[-10:, :10].sum()))
    if r[0]>r[1]:
        i = 0
    else:
        i = 1
    # if chrom=='chr3':
    #     i = 1
    if pearsonr(cpgtmp, pc[:,i])[0]>0:
        pc = pc[:,i]
        modelall.append(pca.components_[i])
    else:
        pc = -pc[:,i]
        modelall.append(-pca.components_[i])
    pcall.append(pc)
    print(chrom, i)


In [21]:
## impute

fig, axes = plt.subplots(2, 22, figsize=(66,4), gridspec_kw={'height_ratios':[5,0.2]}, sharex='col')
for i,c in enumerate(chrom_sizes.index):
    ax = axes[0,i]
    n_bins = int(chrom_sizes[c] // 100000) + 1
    tmp = np.zeros((n_bins, n_bins))
    tmp[np.ix_(binall[i], binall[i])] = Call[i]
    ax.imshow(tmp, cmap='bwr', vmin=np.percentile(Call[i], 5), vmax=np.percentile(Call[i], 95))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(c, fontsize=15)

    ax = axes[1,i]
    # ax.set_title('PC1', fontsize=10)
    sns.despine(bottom=True, left=True, ax=ax)
    tmp = np.zeros(n_bins)
    tmp[binall[i]] = pcall[i]# / np.std(pcall[i])
    x, y = np.arange(n_bins), tmp
    # x, y = np.arange(pcall[i].shape[0]), pcall[i]
    ax.fill_between(x, y, 0, where=y >= 0, facecolor='C3', interpolate=True)
    ax.fill_between(x, y, 0, where=y <= 0, facecolor='C0', interpolate=True)
    ax.set_yticks([])
    ax.set_ylim([np.percentile(y, 1), np.percentile(y, 99)])

plt.tight_layout()
# plt.savefig(f'{indir}/plot/celltype_compraw.pdf', transparent=True)


In [22]:
## raw

fig, axes = plt.subplots(2, 22, figsize=(66,4), gridspec_kw={'height_ratios':[5,0.2]}, sharex='col')
for i,c in enumerate(chrom_sizes.index):
    ax = axes[0,i]
    n_bins = int(chrom_sizes[c] // 100000) + 1
    tmp = np.zeros((n_bins, n_bins))
    tmp[np.ix_(binall[i], binall[i])] = Call[i]
    ax.imshow(tmp, cmap='bwr', vmin=np.percentile(Call[i], 5), vmax=np.percentile(Call[i], 95))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(c, fontsize=15)

    ax = axes[1,i]
    # ax.set_title('PC1', fontsize=10)
    sns.despine(bottom=True, left=True, ax=ax)
    tmp = np.zeros(n_bins)
    tmp[binall[i]] = pcall[i]# / np.std(pcall[i])
    x, y = np.arange(n_bins), tmp
    # x, y = np.arange(pcall[i].shape[0]), pcall[i]
    ax.fill_between(x, y, 0, where=y >= 0, facecolor='C3', interpolate=True)
    ax.fill_between(x, y, 0, where=y <= 0, facecolor='C0', interpolate=True)
    ax.set_yticks([])
    ax.set_ylim([np.percentile(y, 1), np.percentile(y, 99)])

plt.tight_layout()
# plt.savefig(f'{indir}/plot/celltype_compraw.pdf', transparent=True)


In [24]:
bin_df[f'{mode}_binfilter'] = np.concatenate(binall)
bin_df[f'{mode}_pcloading'] = 0
bin_df.loc[bin_df[f'{mode}_binfilter'], f'{mode}_pcloading'] = np.concatenate(modelall)

# np.save(f'{outdir}binfilter_impute.npy', binall)
# np.save(f'{outdir}pcloading_impute.npy', modelall)


In [25]:
selb = bin_df['raw_binfilter'] & bin_df['impute_binfilter']
print(selb.sum())


In [26]:
bin_df.to_hdf(f'{outdir}merged_comp.hdf', key='data')


In [28]:
def compute_comp(cool_path, keep_mat=False):
    pcall = []
    Call = []
    cool = cooler.Cooler(cool_path)
    for k,chrom in enumerate(chrom_sizes.index):
        Q = cool.matrix(balance=False, sparse=True).fetch(chrom).toarray()
        Q = Q - np.diag(np.diag(Q))
        pc = np.zeros(Q.shape[0])
        binfilter = binall[k]
        Q = Q[binfilter][:, binfilter]
        decay = np.array([np.mean(np.diag(Q, i)) for i in range(Q.shape[0])])
        E = np.zeros(Q.shape)
        row, col = np.diag_indices(E.shape[0])
        E[row, col] = 1
        for i in range(1, E.shape[0]):
            E[row[:-i], col[i:]] = (Q[row[:-i], col[i:]] + 1e-5) / (decay[i] + 1e-5)
        E = E + E.T
        C = np.corrcoef(np.log2(E + 0.001))
        vmin, vmax = np.percentile(C, 5), np.percentile(C, 95)
        C[C<vmin] = vmin
        C[C>vmax] = vmax
        pc = (C-np.mean(C, axis=0)).dot(modelall[k])
        pcall.append(pc)
        if keep_mat:
            Call.append(C)
    if keep_mat:
        return Call, pcall
    else:
        return np.concatenate(pcall)


In [29]:
bin_df = pd.read_hdf(f'{outdir}merged_comp.hdf', key='data')
binall = [bin_df.loc[bin_df['chrom']==chrom, 'raw_binfilter'] for chrom in chrom_sizes.index]
modelall = [bin_df.loc[(bin_df['chrom']==chrom), 'raw_pcloading'][binall[k]] for k,chrom in enumerate(chrom_sizes.index)]


In [30]:
## merged impute matrix compartment

Call, pcall = compute_comp(f'{indir_impute}merged.cool', keep_mat=True)

fig, axes = plt.subplots(2, 22, figsize=(66,4), gridspec_kw={'height_ratios':[5,0.2]}, sharex='col')
for i,c in enumerate(chrom_sizes.index):
    ax = axes[0,i]
    n_bins = int(chrom_sizes[c] // 100000) + 1
    tmp = np.zeros((n_bins, n_bins))
    tmp[np.ix_(binall[i], binall[i])] = Call[i]
    ax.imshow(tmp, cmap='bwr', vmin=np.percentile(Call[i], 5), vmax=np.percentile(Call[i], 95), rasterized=True)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(c, fontsize=15)

    ax = axes[1,i]
    # ax.set_title('PC1', fontsize=10)
    sns.despine(bottom=True, left=True, ax=ax)
    tmp = np.zeros(n_bins)
    tmp[binall[i]] = pcall[i]# / np.std(pcall[i])
    x, y = np.arange(n_bins), tmp
    # x, y = np.arange(pcall[i].shape[0]), pcall[i]
    ax.fill_between(x, y, 0, where=y >= 0, facecolor='C3', interpolate=True)
    ax.fill_between(x, y, 0, where=y <= 0, facecolor='C0', interpolate=True)
    ax.set_yticks([])
    ax.set_ylim([np.percentile(y, 1), np.percentile(y, 99)])

plt.tight_layout()
# plt.savefig(f'{indir}/plot/celltype_compraw.pdf', transparent=True)


In [31]:
mode = 'impute'
bin_df[f'{mode}_binfilter'] = np.concatenate(binall)
bin_df[f'{mode}_pcloading'] = 0
bin_df.loc[bin_df[f'{mode}_binfilter'], f'{mode}_pcloading'] = np.concatenate(modelall)

# np.save(f'{outdir}binfilter_impute.npy', binall)
# np.save(f'{outdir}pcloading_impute.npy', modelall)


In [32]:
selb = bin_df['raw_binfilter'] & bin_df['impute_binfilter']
print(selb.sum())


In [33]:
Call, pcall = compute_comp(f'{indir_impute}L1/c34.cool', keep_mat=True)
pc_impute = np.concatenate(pcall)

fig, axes = plt.subplots(2, 22, figsize=(66,4), gridspec_kw={'height_ratios':[5,0.2]}, sharex='col')
for i,c in enumerate(chrom_sizes.index):
    ax = axes[0,i]
    n_bins = int(chrom_sizes[c] // 100000) + 1
    tmp = np.zeros((n_bins, n_bins))
    tmp[np.ix_(binall[i], binall[i])] = Call[i]
    ax.imshow(tmp, cmap='bwr', vmin=np.percentile(Call[i], 5), vmax=np.percentile(Call[i], 95), rasterized=True)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(c, fontsize=15)

    ax = axes[1,i]
    # ax.set_title('PC1', fontsize=10)
    sns.despine(bottom=True, left=True, ax=ax)
    tmp = np.zeros(n_bins)
    tmp[binall[i]] = pcall[i]# / np.std(pcall[i])
    x, y = np.arange(n_bins), tmp
    # x, y = np.arange(pcall[i].shape[0]), pcall[i]
    ax.fill_between(x, y, 0, where=y >= 0, facecolor='C3', interpolate=True)
    ax.fill_between(x, y, 0, where=y <= 0, facecolor='C0', interpolate=True)
    ax.set_yticks([])
    ax.set_ylim([np.percentile(y, 1), np.percentile(y, 99)])

plt.tight_layout()
# plt.savefig(f'{indir}/plot/celltype_compraw.pdf', transparent=True)


In [34]:
Call, pcall = compute_comp(f'{indir_raw}L1/c34.mcool::resolutions/100000', keep_mat=True)
pc_raw = np.concatenate(pcall)

fig, axes = plt.subplots(2, 22, figsize=(66,4), gridspec_kw={'height_ratios':[5,0.2]}, sharex='col')
for i,c in enumerate(chrom_sizes.index):
    ax = axes[0,i]
    n_bins = int(chrom_sizes[c] // 100000) + 1
    tmp = np.zeros((n_bins, n_bins))
    tmp[np.ix_(binall[i], binall[i])] = Call[i]
    ax.imshow(tmp, cmap='bwr', vmin=np.percentile(Call[i], 5), vmax=np.percentile(Call[i], 95), rasterized=True)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(c, fontsize=15)

    ax = axes[1,i]
    # ax.set_title('PC1', fontsize=10)
    sns.despine(bottom=True, left=True, ax=ax)
    tmp = np.zeros(n_bins)
    tmp[binall[i]] = pcall[i]# / np.std(pcall[i])
    x, y = np.arange(n_bins), tmp
    # x, y = np.arange(pcall[i].shape[0]), pcall[i]
    ax.fill_between(x, y, 0, where=y >= 0, facecolor='C3', interpolate=True)
    ax.fill_between(x, y, 0, where=y <= 0, facecolor='C0', interpolate=True)
    ax.set_yticks([])
    ax.set_ylim([np.percentile(y, 1), np.percentile(y, 99)])

plt.tight_layout()
# plt.savefig(f'{indir}/plot/celltype_compraw.pdf', transparent=True)


In [35]:
## c22 Epi-Aci
tmp = bin_df.loc[bin_df['raw_binfilter']]
tmp['raw_pc'] = pc_raw.copy()
tmp['impute_pc'] = pc_impute.copy()

fig, ax = plt.subplots(figsize=(4,4), dpi=300)
sns.scatterplot(data=tmp, x='raw_pc', y='impute_pc', hue='chrom', 
                s=1, edgecolor='none', rasterized=True, ax=ax, palette='tab20')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', markerscale=5, ncol=2)
print(pearsonr(pc_raw, pc_impute))

In [36]:
## c34 Fibro-Myo
tmp = bin_df.loc[bin_df['raw_binfilter']]
tmp['raw_pc'] = pc_raw.copy()
tmp['impute_pc'] = pc_impute.copy()

fig, ax = plt.subplots(figsize=(4,4), dpi=300)
sns.scatterplot(data=tmp, x='raw_pc', y='impute_pc', hue='chrom', 
                s=1, edgecolor='none', rasterized=True, ax=ax, palette='tab20')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', markerscale=5, ncol=2)
print(pearsonr(pc_raw, pc_impute))

In [37]:
def raw_impute_corr(ct):
    Call, pcall = compute_comp(f'{indir_impute}L1/{ct}.cool', keep_mat=True)
    pc_impute = np.concatenate(pcall)
    Call, pcall = compute_comp(f'{indir_raw}L1/{ct}.mcool::resolutions/100000', keep_mat=True)
    pc_raw = np.concatenate(pcall)
    return pearsonr(pc_raw, pc_impute)[0]


In [38]:
cpu = 20
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for ct in L1meta.index:
        future = executor.submit(
            raw_impute_corr,
            ct=ct,
        )
        futures[future] = ct

    result = []
    for future in as_completed(futures):
        ct = futures[future]
        tmp = [ct, L1meta.loc[ct, 'L1_annot'], future.result()]
        result.append(tmp)
        print(f'{ct} finished {tmp}')
        

In [39]:
L1meta = pd.read_csv(f'{ENTEX_ROOT}/L1color.tsv', sep='\t', header=0, index_col=0)
L1meta['L1_annot'] = L1meta['L1_annot'].str.replace(' ','-').str.replace('/','_')

In [40]:
cpu = 20
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for ct in L1meta.index:
        future = executor.submit(
            compute_comp,
            # cool_path=f'{indir_impute}L1/{ct}.cool',
            cool_path=f'{indir_raw}L1/{ct}.mcool::resolutions/100000',
        )
        futures[future] = ct

    result = {}
    for future in as_completed(futures):
        ct = futures[future]
        result[ct] = future.result()
        print(f'{ct} finished')
        

In [41]:
comp = bin_df[['chrom','start','end','raw_binfilter']].copy()
binfilter = bin_df['raw_binfilter'].copy()
for xx in result:
    ct = L1meta.loc[xx, 'L1_annot']
    # tmp = bin_df[['chrom','start','end']].copy()
    comp[xx] = 0
    comp.loc[binfilter, xx] = result[xx]
    comp[['chrom','start','end',xx]].to_csv(f'{outdir}L1/{ct}.raw.bedgraph', sep='\t', header=False, index=False)


In [42]:
comp.to_hdf(f'{outdir}L1/comp.raw.hdf', key='data')


In [44]:
ct = 'c2'
# Call, pcall = compute_comp(f'{indir_impute}L1/{ct}.cool', keep_mat=True)
Call, pcall = compute_comp(f'{indir_raw}L1/{ct}.mcool::resolutions/100000', keep_mat=True)


In [45]:
from scipy.sparse import coo_matrix
from schicluster.cool import get_chrom_offsets

def chrom_sum_iterator(Call,
                       binall,
                       chrom_sizes,
                       chrom_offset
                      ):
    for i,c in enumerate(chrom_sizes.index):
        n_bins = int(chrom_sizes[c] // 100000) + 1
        tmp = np.zeros((n_bins, n_bins))
        tmp[np.ix_(binall[i], binall[i])] = Call[i]
        matrix = coo_matrix(np.triu(tmp))
        pixel_df = pd.DataFrame({
            'bin1_id': matrix.row,
            'bin2_id': matrix.col,
            'count': matrix.data
        })
        pixel_df.iloc[:, :2] += chrom_offset[c]
        yield pixel_df

bin_tmp = bin_df.reset_index()
chrom_offset = {c: bin_tmp[bin_tmp['chrom'] == c].index[0] for c in chrom_sizes.index}
chrom_iter = chrom_sum_iterator(Call,
                                binall,
                                chrom_sizes,
                                chrom_offset
                               )
cooler.create_cooler(cool_uri=f'{outdir}{ct}.raw.corr.cool',
                     bins=bin_df[['chrom', 'start', 'end']],
                     pixels=chrom_iter,
                     ordered=True,
                     dtypes={'count': np.float32})


In [46]:
ct

## Saddle plot of merged raw pc transformed compartment

In [50]:
leg = ['L23_IT', 'L4_IT', 'L56_NP', 'L5_ET', 'L5_IT', 'L6_CT', 'L6_IT', 'L6_IT_Car3', 'L6b', 
       'Lamp5', 'Lamp5_LHX6', 'Sncg', 'Vip', 'Pvalb', 'Pvalb_ChC', 'Sst', 
       'MSN_D1', 'MSN_D2', 'SubCtx', 'Amy', 'CHD7', 'Foxp2', 
       'ASC', 'ODC', 'OPC', 'MGC', 'EC', 'PC', 'VLMC', 'merged']
legname = ['L2/3-IT', 'L4-IT', 'L5/6-NP', 'L5-ET', 'L5-IT', 'L6-CT', 'L6-IT', 'L6-IT-Car3', 'L6b', 
       'Lamp5', 'Lamp5-Lhx6', 'Sncg', 'Vip', 'Pvalb', 'Pvalb-ChC', 'Sst', 
       'MSN-D1', 'MSN-D2', 'SubCtx-Cplx', 'Amy-Exc', 'Chd7', 'Foxp2', 
       'ASC', 'ODC', 'OPC', 'MGC', 'EC', 'PC', 'VLMC', 'merged'
      ]


In [51]:
# binall = np.load(f'{outdir}binfilter_raw.npy', allow_pickle=True)
# modelall = np.load(f'{outdir}pcloading_raw.npy', allow_pickle=True)


In [52]:
def compsaddle(cool):
    sad = np.zeros((50, 50))
    count = np.zeros((50, 50))
    comptmp = []
    for k,chrom in enumerate(chrom_sizes.index[:-1]):
        Q = cool.matrix(balance=False, sparse=True).fetch(chrom).toarray()
        Q = Q - np.diag(np.diag(Q))
        pc = np.zeros(Q.shape[0])
        binfilter = binall[k]
        Q = Q[binfilter][:, binfilter]
        decay = np.array([np.mean(np.diag(Q, i)) for i in range(Q.shape[0])])
        E = np.zeros(Q.shape)
        row, col = np.diag_indices(E.shape[0])
        E[row, col] = 1
        for i in range(1, E.shape[0]):
            E[row[:-i], col[i:]] = (Q[row[:-i], col[i:]] + 1e-5) / (decay[i] + 1e-5)
        E = E + E.T
        C = np.corrcoef(np.log2(E + 0.001))
        pc = (C-np.mean(C, axis=0)).dot(modelall[k])
        comptmp.append(pc)
        labels, groups = pd.qcut(pc, 50, labels=False, retbins=True)
        sad += np.array([[E[np.ix_(labels==i, labels==j)].sum() for i in range(50)] for j in range(50)])
        count += np.array([[(labels==i).sum()*(labels==j).sum() for i in range(50)] for j in range(50)])
    return np.concatenate(comptmp), sad, count


In [53]:
mode = 'raw'
sad = np.zeros((len(leg),50,50))
count = np.zeros((len(leg),50,50))
comp = []


In [54]:
cpu = 10
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for t,ct in enumerate(leg):
        if mode=='impute':
            cool = cooler.Cooler(f'{indir}{ct}/{ct}.Q.cool')
        elif mode=='raw':
            cool = cooler.Cooler(f'{indir}{ct}/{ct}/{ct}.raw.mcool::resolutions/100000')
        future = executor.submit(
            compsaddle,
            cool=cool,
        )
        futures[future] = t

    for future in as_completed(futures):
        t = futures[future]
        ct = leg[t]
        xx, yy, zz = future.result()
        comp.append(pd.Series(xx, name=ct))
        sad[t] += yy
        count[t] += zz
        print(f'{ct} finished')
        

In [55]:
bins_df = cool.bins()[:]
bins_df.index = bins_df['chrom'].astype(str) + '-' + (bins_df['start'] // res).astype(str)
compidx = np.concatenate([bins_df.index[bins_df['chrom']==c][binall[i]] for i,c in enumerate(chrom_sizes.index[:-1])])


In [56]:
comp = pd.concat(comp, axis=1)
comp.index = compidx
comp.to_hdf(f'{outdir}comp_{mode}_mergerawpca.hdf', key='data')


In [57]:
sad = sad / count
np.save(f'{outdir}saddle_{mode}_mergerawpca.npy', sad)


In [58]:
ws = 10
compstr = [[sad[i][:ws, :ws].mean(), sad[i][-ws:, -ws:].mean(), sad[i][:ws, -ws:].mean(), sad[i][-ws:, :ws].mean(), 
            (sad[i][:ws, :ws].sum() + sad[i][-ws:, -ws:].sum()) / (sad[i][:ws, -ws:].sum() + sad[i][-ws:, :ws].sum())]
            for i in range(len(leg))]
compstr = pd.DataFrame(compstr, index=leg, columns=['BB', 'AA', 'BA', 'AB', 'strength'])
compstr.sort_values('strength')
